In [ ]:
# ============================================================================
# HISTORICAL RESULT RECOVERY - ETTm1 -> ETTh1, SSA, H=192
# Using the SAME implementation that produced the old zero-shot table
# ============================================================================

import os
import gc
import json
import time
import random
import warnings
import subprocess
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional
import sys

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# ============================================================================
# DEVICE SETUP
# ============================================================================
print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")

# ============================================================================
# REPRODUCIBILITY
# ============================================================================
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

SEED = 2024
set_seed(SEED)
print(f"Seed: {SEED}")

# ============================================================================
# SSA RECONSTRUCTION (EXACT COPY FROM OLD CODE)
# ============================================================================

class SSARecovery:
    @staticmethod
    def _embed(signal: np.ndarray, L: int) -> np.ndarray:
        N = len(signal)
        K = N - L + 1
        X = np.zeros((L, K), dtype=np.float32)
        for i in range(L):
            X[i, :] = signal[i:i + K]
        return X

    @staticmethod
    def _diagonal_averaging(X: np.ndarray) -> np.ndarray:
        L, K = X.shape
        N = L + K - 1
        signal = np.zeros(N, dtype=np.float32)
        counts = np.zeros(N, dtype=np.float32)
        for i in range(L):
            for j in range(K):
                signal[i + j] += X[i, j]
                counts[i + j] += 1.0
        return signal / np.maximum(counts, 1e-8)

    @staticmethod
    def denoise_signal(signal: np.ndarray, L: int = 30, r: int = 5) -> np.ndarray:
        signal = signal.copy()
        N = len(signal)

        if N < L:
            L = max(2, N // 2)

        if np.any(np.isnan(signal)):
            mask = np.isnan(signal)
            if np.any(~mask):
                signal[mask] = np.interp(np.flatnonzero(mask), np.flatnonzero(~mask), signal[~mask])
            else:
                return signal.astype(np.float32)

        if np.std(signal) < 1e-8:
            return signal.astype(np.float32)

        try:
            X = SSARecovery._embed(signal, L)
            U, S, Vt = np.linalg.svd(X, full_matrices=False)
            r = min(r, len(S))
            Xr = U[:, :r] @ np.diag(S[:r]) @ Vt[:r, :]
            out = SSARecovery._diagonal_averaging(Xr)[:N]
            return out.astype(np.float32)
        except Exception:
            return signal.astype(np.float32)

    @staticmethod
    def apply_to_multivariate(data: np.ndarray, L: int = 30, r: int = 5) -> np.ndarray:
        out = np.zeros_like(data, dtype=np.float32)
        for c in range(data.shape[1]):
            out[:, c] = SSARecovery.denoise_signal(data[:, c], L, r)
        return out


# ============================================================================
# DATA LOADING (EXACT COPY FROM OLD CODE)
# ============================================================================

class DataManager:
    DATASET_URLS = {
        "ETTh1": "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv",
        "ETTh2": "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh2.csv",
        "ETTm1": "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTm1.csv",
        "ETTm2": "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTm2.csv",
    }

    @staticmethod
    def load_raw(name: str) -> np.ndarray:
        url = DataManager.DATASET_URLS[name]
        print(f"Loading {name} from {url}...")

        df = pd.read_csv(url)
        if "date" in df.columns:
            df = df.drop(columns=["date"])

        values = df.values.astype(np.float32)
        print(f"✅ Loaded {name}: {values.shape[1]} channels, {values.shape[0]} time steps")
        return values


class WindowDataset(Dataset):
    def __init__(self, data: np.ndarray, seq_len: int, pred_len: int):
        self.data = torch.tensor(data, dtype=torch.float32)
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.window_len = seq_len + pred_len

        if len(data) >= self.window_len:
            self.n_windows = len(data) - self.window_len + 1
        else:
            self.n_windows = 0

    def __len__(self):
        return self.n_windows

    def __getitem__(self, idx):
        window = self.data[idx:idx + self.window_len]
        x_enc = window[:self.seq_len]
        y = window[self.seq_len:self.seq_len + self.pred_len]
        x_mark_dec_dummy = torch.zeros(self.seq_len + self.pred_len, 1)
        return x_enc, y, x_mark_dec_dummy


# ============================================================================
# ICTSP CORE (EXACT COPY FROM OLD CODE)
# ============================================================================

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_size, heads, dropout=0.1):
        super().__init__()
        self.embed_size = embed_size
        self.heads = heads
        self.head_dim = embed_size // heads
        self.dropout = nn.Dropout(dropout)
        assert self.head_dim * heads == embed_size
        self.values = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.keys = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.queries = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.fc_out = nn.Linear(heads * self.head_dim, embed_size)

    def forward(self, values, keys, queries, mask=None):
        N = queries.shape[0]
        value_len, key_len, query_len = values.shape[1], keys.shape[1], queries.shape[1]
        values = values.reshape(N, value_len, self.heads, self.head_dim)
        keys = keys.reshape(N, key_len, self.heads, self.head_dim)
        queries = queries.reshape(N, query_len, self.heads, self.head_dim)
        values = self.values(values)
        keys = self.dropout(self.keys(keys))
        queries = self.queries(queries)
        energy = torch.einsum("nqhd,nkhd->nhqk", [queries, keys])
        if mask is not None:
            energy = energy.masked_fill(mask == 0, float("-1e20"))
        energy = energy / (self.embed_size ** 0.5)
        attention = F.softmax(energy, dim=-1)
        out = torch.einsum("nhql,nlhd->nqhd", [attention, values]).reshape(
            N, query_len, self.heads * self.head_dim
        )
        out = self.fc_out(out)
        return out, attention


class TransformerEncoder(nn.Module):
    def __init__(self, emb_size=128, depth=3, heads=8, mlp_ratio=4, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=emb_size,
                nhead=heads,
                dim_feedforward=mlp_ratio * emb_size,
                batch_first=True,
                dropout=dropout,
                norm_first=False,
                activation="gelu",
            )
            for _ in range(depth)
        ])

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x


class Tokenizer(nn.Module):
    def __init__(self, lookback=512, output=96, stride=8):
        super().__init__()
        self.d = lookback + output
        self.s = stride

    def forward(self, tensor):
        B, C, L = tensor.shape
        if L >= self.d:
            unfolded = tensor.flip(-1).unfold(dimension=2, size=self.d, step=self.s)
            return unfolded.flip(-1).flip(-2)
        else:
            padding = torch.zeros(B, C, self.d - L, device=tensor.device, dtype=tensor.dtype)
            tensor_padded = torch.cat([tensor, padding], dim=-1)
            unfolded = tensor_padded.flip(-1).unfold(dimension=2, size=self.d, step=self.s)
            return unfolded.flip(-1).flip(-2)


class ICTSP(nn.Module):
    def __init__(
        self,
        lookback=512,
        output=96,
        depth=3,
        heads=8,
        mlp_ratio=4,
        d_model=128,
        external_stride=8,
        n_channels=7,
        dropout=0.5,
        token_retriever_flag=True,
        token_limit=2048,
    ):
        super().__init__()
        self.lookback = lookback
        self.pred_len = output
        self.external_stride = external_stride
        self.input_projection = nn.Linear(lookback + output, d_model)
        self.transformer_encoder = TransformerEncoder(d_model, depth, heads, mlp_ratio, dropout)
        self.input_norm = nn.LayerNorm(d_model)
        self.output_norm = nn.LayerNorm(d_model)
        self.output_embedding = nn.Parameter(0.01 * torch.randn(1, 1, 1200))
        self.output_projection = nn.Linear(d_model, output)
        self.channel_embedding = nn.Parameter(0.01 * torch.randn(1, 1024, d_model))
        self.pos_embedding = nn.Parameter(0.01 * torch.randn(1, 8192, 1, d_model))
        self.pos_embedding_after = nn.Parameter(0.01 * torch.randn(1, 8192, d_model))
        self.token_retriever_flag = token_retriever_flag
        self.linear_warmup_steps = 5000
        self.linear_warm_up_counter = 0
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)

    def forward(self, x, x_mark_dec=None):
        lookback = self.lookback
        future = self.pred_len
        mean = x[:, [-1], :].detach()
        x = x.permute(0, 2, 1)
        output_embedding = self.output_embedding[:, :, 0:future].expand(x.shape[0], x.shape[1], -1)
        x = torch.cat([x, output_embedding + mean.permute(0, 2, 1)], dim=-1)
        B, C, _ = x.shape
        number_of_targets = C
        x_orig = x[:, :, 0:-future].clone()
        external_tokenizer = Tokenizer(lookback, future, stride=self.external_stride)
        ex_tokens = external_tokenizer(x_orig)
        _, _, _, d = ex_tokens.shape
        ex_tokens = ex_tokens.permute(0, 2, 1, 3).reshape(B, -1, d)
        x_target = x[:, -number_of_targets:, -(lookback + future):]
        x_tokens = torch.cat([ex_tokens, x_target], dim=1)
        token_mean = x_tokens[:, :, [-(future + 1)]].detach()
        x_tokens = x_tokens - token_mean
        x_tokens = self.input_projection(x_tokens)
        channel_mask = self.channel_embedding[:, -C:, :]
        x_tokens = x_tokens + channel_mask.repeat(1, x_tokens.shape[1] // C, 1)
        pos_emb = self.pos_embedding[:, -(x_tokens.shape[1] // C):, :, :].expand(-1, -1, C, -1)
        pos_emb = pos_emb.reshape(pos_emb.shape[0], pos_emb.shape[1] * pos_emb.shape[2], pos_emb.shape[3])
        x_tokens = x_tokens + pos_emb

        if self.linear_warm_up_counter < self.linear_warmup_steps:
            if self.training:
                self.linear_warm_up_counter += 1
            x_output = self.output_projection(x_tokens[:, -number_of_targets:, :])
            x_output = x_output + token_mean[:, -x_output.shape[1]:, :]
            x_output = x_output.permute(0, 2, 1)
            return x_output

        x_tokens = self.input_norm(x_tokens)
        x_tokens = self.transformer_encoder(x_tokens)
        x_output = x_tokens[:, -number_of_targets:, :]
        x_output = self.output_norm(x_output)
        x_output = self.output_projection(x_output)
        x_output = x_output + token_mean[:, -x_output.shape[1]:, :]
        x_output = x_output.permute(0, 2, 1)
        return x_output


class RepoICTSPModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.model = ICTSP(
            lookback=cfg.lookback,
            output=cfg.pred_len,
            depth=cfg.e_layers,
            heads=cfg.n_heads,
            mlp_ratio=cfg.mlp_ratio,
            d_model=cfg.d_model,
            external_stride=cfg.sampling_step,
            n_channels=cfg.enc_in,
            dropout=cfg.dropout,
        )

    def forward(self, x, x_mark_dec):
        return self.model(x, x_mark_dec)


# ============================================================================
# METRICS & EVALUATION
# ============================================================================

def compute_metrics(pred, target):
    with torch.no_grad():
        mse = F.mse_loss(pred, target).item()
        mae = F.l1_loss(pred, target).item()
    return mse, mae


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_mse = 0.0
    total_mae = 0.0
    n_batches = 0
    n_samples = 0

    for x_enc, y, x_mark_dec in loader:
        x_enc = x_enc.to(device)
        y = y.to(device)
        x_mark_dec = x_mark_dec.to(device)
        pred = model(x_enc, x_mark_dec)
        mse, mae = compute_metrics(pred, y)
        total_mse += mse * x_enc.shape[0]
        total_mae += mae * x_enc.shape[0]
        n_batches += 1
        n_samples += x_enc.shape[0]

    return {
        "mse": total_mse / n_samples if n_samples > 0 else float("inf"),
        "mae": total_mae / n_samples if n_samples > 0 else float("inf"),
    }


# ============================================================================
# CONFIGURATION
# ============================================================================

@dataclass
class Config:
    source_name: str = "ETTm1"
    target_name: str = "ETTh1"
    use_ssa: bool = True
    lookback: int = 512
    pred_len: int = 192
    e_layers: int = 3
    d_model: int = 128
    n_heads: int = 8
    mlp_ratio: int = 4
    dropout: float = 0.5
    sampling_step: int = 8
    batch_size: int = 32
    batch_size_test: int = 32
    learning_rate: float = 5e-4
    weight_decay: float = 1e-5
    train_epochs: int = 100
    patience_steps: int = 20
    grad_clip: float = 1.0
    enc_in: int = 7
    ssa_L: int = 30
    ssa_r: int = 5
    seed: int = 2024


# ============================================================================
# HISTORICAL RECOVERY TRAINER
# ============================================================================

class HistoricalRecoveryTrainer:
    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.source_name = cfg.source_name
        self.target_name = cfg.target_name
        set_seed(cfg.seed)

        print(f"\n{'='*70}")
        print(f"HISTORICAL RESULT RECOVERY")
        print(f"{self.source_name} → {self.target_name}")
        print(f"Method: SSA | pred_len={cfg.pred_len}")
        print(f"{'='*70}")

    def apply_ssa(self, data: np.ndarray) -> np.ndarray:
        print(f"  Applying SSA reconstruction (L={self.cfg.ssa_L}, r={self.cfg.ssa_r})")
        return SSARecovery.apply_to_multivariate(data, self.cfg.ssa_L, self.cfg.ssa_r)

    def prepare_data(self):
        print(f"\n[1/4] Loading data...")

        # Load source and target
        source_values = DataManager.load_raw(self.source_name)
        target_values = DataManager.load_raw(self.target_name)

        print(f"  Source {self.source_name}: {source_values.shape}")
        print(f"  Target {self.target_name}: {target_values.shape}")
        assert source_values.shape[1] == 7, f"Source should have 7 channels, got {source_values.shape[1]}"
        assert target_values.shape[1] == 7, f"Target should have 7 channels, got {target_values.shape[1]}"

        # Apply SSA reconstruction to FULL series (historical protocol)
        print(f"\n[2/4] Applying SSA reconstruction...")
        source_reconstructed = self.apply_ssa(source_values)
        target_reconstructed = self.apply_ssa(target_values)

        # Update config with source channels
        self.cfg.enc_in = source_reconstructed.shape[1]
        print(f"  Channels: source={self.cfg.enc_in}, target={target_reconstructed.shape[1]}")

        # Create splits for source (70% train, 10% val, 20% unused)
        n_total = len(source_reconstructed)
        n_train = int(n_total * 0.70)
        n_val = int(n_total * 0.10)

        train_data = source_reconstructed[:n_train]
        val_data = source_reconstructed[n_train:n_train + n_val]

        print(f"  Source train samples: {len(train_data)}")
        print(f"  Source val samples: {len(val_data)}")

        # Standardize using source data only
        print(f"\n[3/4] Standardizing using source statistics...")
        self.scaler = StandardScaler()
        self.scaler.fit(train_data)

        train_scaled = self.scaler.transform(train_data).astype(np.float32)
        val_scaled = self.scaler.transform(val_data).astype(np.float32)

        # Apply SAME scaler to target (zero-shot!)
        target_scaled = self.scaler.transform(target_reconstructed).astype(np.float32)

        # Use last 30% of target for zero-shot testing
        n_test_start = int(len(target_scaled) * 0.70)
        test_data = target_scaled[n_test_start:]

        print(f"  Target test samples: {len(test_data)}")

        # Create datasets
        self.train_ds = WindowDataset(train_scaled, self.cfg.lookback, self.cfg.pred_len)
        self.val_ds = WindowDataset(val_scaled, self.cfg.lookback, self.cfg.pred_len)
        self.test_ds = WindowDataset(test_data, self.cfg.lookback, self.cfg.pred_len)

        self.train_loader = DataLoader(self.train_ds, batch_size=self.cfg.batch_size, shuffle=True)
        self.val_loader = DataLoader(self.val_ds, batch_size=self.cfg.batch_size_test, shuffle=False)
        self.test_loader = DataLoader(self.test_ds, batch_size=self.cfg.batch_size_test, shuffle=False)

        print(f"  Train windows: {len(self.train_ds)}")
        print(f"  Val windows: {len(self.val_ds)}")
        print(f"  Test windows: {len(self.test_ds)}")

    def train(self):
        print(f"\n[4/4] Training on {self.source_name}...")

        self.model = RepoICTSPModel(self.cfg).to(self.device)
        total_params = sum(p.numel() for p in self.model.parameters())
        print(f"  Model parameters: {total_params:,}")

        # Use Adam optimizer (matching old code)
        optimizer = torch.optim.Adam(
            self.model.parameters(),
            lr=self.cfg.learning_rate,
            weight_decay=self.cfg.weight_decay,
        )

        best_val_mse = float("inf")
        best_state = None
        patience_counter = 0

        print(f"\n  Epoch | Train Loss | Val MSE | Best Val MSE")
        print(f"  {'-'*50}")

        for epoch in range(1, self.cfg.train_epochs + 1):
            self.model.train()
            epoch_loss = 0.0
            n_batches = 0

            for x_enc, y, x_mark_dec in self.train_loader:
                x_enc = x_enc.to(self.device)
                y = y.to(self.device)
                x_mark_dec = x_mark_dec.to(self.device)

                optimizer.zero_grad()
                pred = self.model(x_enc, x_mark_dec)
                loss = F.mse_loss(pred, y)
                loss.backward()

                if self.cfg.grad_clip > 0:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.grad_clip)

                optimizer.step()
                epoch_loss += loss.item()
                n_batches += 1

            # Validation
            val_metrics = evaluate(self.model, self.val_loader, self.device)

            if val_metrics["mse"] < best_val_mse - 1e-8:
                best_val_mse = val_metrics["mse"]
                best_state = {k: v.detach().cpu().clone() for k, v in self.model.state_dict().items()}
                patience_counter = 0
                print(f"  {epoch:4d}  | {epoch_loss/n_batches:.6f} | {val_metrics['mse']:.6f} | {best_val_mse:.6f}  ✓")
            else:
                patience_counter += 1
                if epoch % 10 == 0 or epoch == 1:
                    print(f"  {epoch:4d}  | {epoch_loss/n_batches:.6f} | {val_metrics['mse']:.6f} | {best_val_mse:.6f}")

            if patience_counter >= self.cfg.patience_steps:
                print(f"\n  Early stopping at epoch {epoch}")
                break

        if best_state is not None:
            self.model.load_state_dict(best_state)

        print(f"\n  Best Validation MSE: {best_val_mse:.6f}")
        return best_val_mse

    def evaluate_zero_shot(self):
        print(f"\n  Zero-shot evaluation on {self.target_name}...")
        test_metrics = evaluate(self.model, self.test_loader, self.device)
        print(f"  Zero-Shot MSE: {test_metrics['mse']:.6f}")
        print(f"  Zero-Shot MAE: {test_metrics['mae']:.6f}")
        return test_metrics

    def save_results(self, best_val_mse, test_metrics):
        # Save JSON
        results = {
            "source": self.source_name,
            "target": self.target_name,
            "method": "SSA",
            "pred_len": self.cfg.pred_len,
            "lookback": self.cfg.lookback,
            "SSA_L": self.cfg.ssa_L,
            "SSA_r": self.cfg.ssa_r,
            "seed": self.cfg.seed,
            "best_validation_mse": best_val_mse,
            "zero_shot_mse": test_metrics["mse"],
            "zero_shot_mae": test_metrics["mae"],
            "training_samples": len(self.train_ds.data),
            "validation_samples": len(self.val_ds.data),
            "target_test_samples": len(self.test_ds.data),
            "training_windows": len(self.train_ds),
            "validation_windows": len(self.val_ds),
            "test_windows": len(self.test_ds),
            "source_channels": self.cfg.enc_in,
            "target_channels": 7,
        }

        with open("historical_ETTm1_to_ETTh1_SSA_H192.json", "w") as f:
            json.dump(results, f, indent=2)

        # Save CSV
        df = pd.DataFrame([results])
        df.to_csv("historical_ETTm1_to_ETTh1_SSA_H192.csv", index=False)

        print(f"\n  Saved results to:")
        print(f"    - historical_ETTm1_to_ETTh1_SSA_H192.json")
        print(f"    - historical_ETTm1_to_ETTh1_SSA_H192.csv")

    def save_checkpoint(self, best_val_mse):
        # Save best model checkpoint
        checkpoint = {
            "model_state_dict": self.model.state_dict(),
            "cfg": self.cfg,
            "best_val_mse": best_val_mse,
            "source": self.source_name,
            "target": self.target_name,
            "method": "SSA",
            "pred_len": self.cfg.pred_len,
            "seed": self.cfg.seed,
        }
        torch.save(checkpoint, "historical_ETTm1_to_ETTh1_SSA_H192_best.pt")
        print(f"  Saved checkpoint: historical_ETTm1_to_ETTh1_SSA_H192_best.pt")

    def run(self):
        self.prepare_data()
        best_val_mse = self.train()
        test_metrics = self.evaluate_zero_shot()
        self.save_results(best_val_mse, test_metrics)
        self.save_checkpoint(best_val_mse)
        return best_val_mse, test_metrics


# ============================================================================
# MAIN
# ============================================================================

if __name__ == "__main__":
    print("\n" + "=" * 70)
    print("HISTORICAL RESULT RECOVERY")
    print("ETTm1 → ETTh1 | SSA | H=192")
    print("=" * 70)
    print(f"Seed: {SEED}")
    print(f"SSA L: 30, SSA r: 5")
    print("=" * 70)

    # Create config
    cfg = Config(
        source_name="ETTm1",
        target_name="ETTh1",
        use_ssa=True,
        pred_len=192,
    )

    # Run experiment
    trainer = HistoricalRecoveryTrainer(cfg)
    best_val_mse, test_metrics = trainer.run()

    # Final output
    print("\n" + "=" * 70)
    print("HISTORICAL RESULT VERIFICATION")
    print("=" * 70)
    print(f"Source          : ETTm1")
    print(f"Target          : ETTh1")
    print(f"Method          : SSA")
    print(f"Prediction H    : 192")
    print(f"SSA L           : 30")
    print(f"SSA r           : 5")
    print(f"Seed            : {SEED}")
    print(f"Best Val MSE    : {best_val_mse:.6f}")
    print(f"Zero-Shot MSE   : {test_metrics['mse']:.6f}")
    print(f"Zero-Shot MAE   : {test_metrics['mae']:.6f}")
    print("=" * 70)
    print("\n✅ Historical recovery experiment completed!")
    print("   - historical_ETTm1_to_ETTh1_SSA_H192.json")
    print("   - historical_ETTm1_to_ETTh1_SSA_H192.csv")
    print("   - historical_ETTm1_to_ETTh1_SSA_H192_best.pt")

Python version: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
PyTorch version: 2.11.0+cu128
CUDA available: True
Using device: cuda
GPU name: Tesla T4
Seed: 2024

HISTORICAL RESULT RECOVERY
ETTm1 → ETTh1 | SSA | H=192
Seed: 2024
SSA L: 30, SSA r: 5

HISTORICAL RESULT RECOVERY
ETTm1 → ETTh1
Method: SSA | pred_len=192

[1/4] Loading data...
Loading ETTm1 from https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTm1.csv...
✅ Loaded ETTm1: 7 channels, 69680 time steps
Loading ETTh1 from https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv...
✅ Loaded ETTh1: 7 channels, 17420 time steps
  Source ETTm1: (69680, 7)
  Target ETTh1: (17420, 7)

[2/4] Applying SSA reconstruction...
  Applying SSA reconstruction (L=30, r=5)
  Applying SSA reconstruction (L=30, r=5)
  Channels: source=7, target=7
  Source train samples: 48776
  Source val samples: 6968

[3/4] Standardizing using source statistics...
  Target test samples: 5226
  Train windows: 48073